# 04 — Model Evaluation & Comparison

This notebook:
1. Loads both saved models
2. Evaluates each on the **test set only** (no generator leakage)
3. Computes Accuracy, Precision, Recall, F1, AUC
4. Plots Confusion Matrices and ROC Curves
5. Produces a side-by-side comparison table saved to CSV

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_curve, auc)

ROOT       = r"D:\Hams Khaled\CS Year 3 Sem 2\Image Processing\Final Project\project"
SPLIT_DIR  = os.path.join(ROOT, "data",    "split")
MODELS_DIR = os.path.join(ROOT, "models")
PLOTS_DIR  = os.path.join(ROOT, "results", "plots")
RESULTS_DIR= os.path.join(ROOT, "results")

os.makedirs(PLOTS_DIR, exist_ok=True)

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
print("  ✔ Paths configured.")


## Step 1 — Load Both Models

In [ ]:
scratch_path  = os.path.join(MODELS_DIR, "scratch_model.h5")
transfer_path = os.path.join(MODELS_DIR, "transfer_model.h5")

scratch_model  = load_model(scratch_path)
transfer_model = load_model(transfer_path)
print("  ✔ scratch_model  loaded from:", scratch_path)
print("  ✔ transfer_model loaded from:", transfer_path)


## Step 2 — Test Generator (shuffle=False)

Using `shuffle=False` so that predicted labels align with true labels in order.

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_gen = test_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, "test"),
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = "binary",
    shuffle     = False,
)
print(f"  Test samples : {test_gen.samples}")
print(f"  Class indices: {test_gen.class_indices}")

y_true = test_gen.classes   # ground-truth labels


## Step 3 — Compute Predictions & Metrics

In [ ]:
def evaluate_model(model, gen, y_true, name):
    """Returns dict of metrics and raw predictions."""
    print(f"\n  ▶ Evaluating {name} …")
    gen.reset()
    y_prob = model.predict(gen, verbose=1).flatten()
    y_pred = (y_prob >= 0.5).astype(int)

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score   = auc(fpr, tpr)
    cm          = confusion_matrix(y_true, y_pred)

    print(f"    Accuracy  : {acc:.4f}")
    print(f"    Precision : {prec:.4f}")
    print(f"    Recall    : {rec:.4f}")
    print(f"    F1-Score  : {f1:.4f}")
    print(f"    AUC       : {auc_score:.4f}")
    return {"acc": acc, "prec": prec, "rec": rec, "f1": f1,
            "auc": auc_score, "cm": cm, "fpr": fpr, "tpr": tpr, "y_prob": y_prob}

scratch_metrics  = evaluate_model(scratch_model,  test_gen, y_true, "CNN Scratch")
transfer_metrics = evaluate_model(transfer_model, test_gen, y_true, "Transfer MobileNetV2")


## Step 4 — Confusion Matrices

In [ ]:
def plot_cm(cm, title, out_path):
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Fresh', 'Rotten'],
                yticklabels=['Fresh', 'Rotten'], ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(title)
    fig.tight_layout()
    plt.savefig(out_path, dpi=150); plt.close()
    print(f"  ✔ Saved → {out_path}")

plot_cm(scratch_metrics["cm"],
        "CNN Scratch — Confusion Matrix",
        os.path.join(PLOTS_DIR, "scratch_confusion.png"))

plot_cm(transfer_metrics["cm"],
        "Transfer MobileNetV2 — Confusion Matrix",
        os.path.join(PLOTS_DIR, "transfer_confusion.png"))


## Step 5 — ROC Curves

In [ ]:
def plot_roc(fpr, tpr, auc_score, title, out_path):
    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, 'b-', lw=2, label=f'ROC (AUC = {auc_score:.4f})')
    plt.plot([0, 1], [0, 1], 'r--', lw=1, label='Random Classifier')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(title); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150); plt.close()
    print(f"  ✔ Saved → {out_path}")

plot_roc(scratch_metrics["fpr"],  scratch_metrics["tpr"],  scratch_metrics["auc"],
         "CNN Scratch — ROC Curve",
         os.path.join(PLOTS_DIR, "scratch_roc.png"))

plot_roc(transfer_metrics["fpr"], transfer_metrics["tpr"], transfer_metrics["auc"],
         "Transfer MobileNetV2 — ROC Curve",
         os.path.join(PLOTS_DIR, "transfer_roc.png"))


## Step 6 — Comparison Table

In [ ]:
# Load training metadata saved by earlier notebooks
def load_meta(path, default={}):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return default

scratch_meta  = load_meta(os.path.join(RESULTS_DIR, "scratch_meta.json"))
transfer_meta = load_meta(os.path.join(RESULTS_DIR, "transfer_meta.json"))

s_params  = scratch_meta.get("scratch_params",          "N/A")
t_params  = transfer_meta.get("transfer_params_p2",     "N/A")
s_time    = scratch_meta.get("scratch_training_time",   "N/A")
t_time    = transfer_meta.get("transfer_training_time", "N/A")
s_epochs  = scratch_meta.get("scratch_epochs",          "N/A")
t_epochs  = transfer_meta.get("transfer_epochs",        "N/A")

comparison = {
    "Model"               : ["CNN Scratch", "Transfer MobileNetV2"],
    "Test Accuracy"       : [round(scratch_metrics["acc"],  4), round(transfer_metrics["acc"],  4)],
    "Test Precision"      : [round(scratch_metrics["prec"], 4), round(transfer_metrics["prec"], 4)],
    "Test Recall"         : [round(scratch_metrics["rec"],  4), round(transfer_metrics["rec"],  4)],
    "Test F1-Score"       : [round(scratch_metrics["f1"],   4), round(transfer_metrics["f1"],   4)],
    "AUC Score"           : [round(scratch_metrics["auc"],  4), round(transfer_metrics["auc"],  4)],
    "Training Time (s)"   : [round(s_time, 1) if isinstance(s_time, float) else s_time,
                              round(t_time, 1) if isinstance(t_time, float) else t_time],
    "Trainable Params"    : [s_params, t_params],
    "Epochs Run"          : [s_epochs, t_epochs],
}

df = pd.DataFrame(comparison)
df = df.set_index("Model")
print("\n", df.to_string())

csv_path = os.path.join(RESULTS_DIR, "comparison_table.csv")
df.to_csv(csv_path)
print(f"\n  ✔ Comparison table saved → {csv_path}")


## Step 7 — Redisplay Loss & Accuracy Curves

In [ ]:
from IPython.display import Image as IPImage, display

for name, suffix in [("CNN Scratch", "scratch"), ("Transfer MobileNetV2", "transfer")]:
    for metric in ["accuracy", "loss"]:
        path = os.path.join(PLOTS_DIR, f"{suffix}_{metric}.png")
        if os.path.exists(path):
            print(f"  {name} — {metric.capitalize()} curve:")
            display(IPImage(filename=path, width=600))
        else:
            print(f"  [WARN] Not found: {path}")

print("\n  ✔ Evaluation complete!")
